[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_dae_top3_seed1.ipynb)

# Part A: the top 3 DAEs for 100 epochs, seed 1

`history/colab_dae_top3.ipynb` with `PART = 2`: the top 3 of `history/colab_dae_top3.ipynb` (read from Drive) for 100 epochs with seed 1, 3 runs of about 55 minutes. Run it next to `colab_dae_top3_seed2.ipynb`, in its own runtime. Stage A was done in `history/colab_dae_top3.ipynb` and is skipped here.

The last cell shows every finished run of the 9; when all 9 are done it saves the best mean val k-NN as `"dae"`.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
Every run writes to `midterm_runs`, next to the earlier runs. The runs here have `tune_pretrain_DAE` in their folder name.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"

## TensorBoard
Filter the runs with `tune_pretrain`, then *Toggle all runs*. `epoch/val_knn_top1` shows where each run peaks, `epoch/val_loss` the reconstruction, the TEXT tab each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## The runs
`train` runs train.py only if the run has no `results.json` yet: after a disconnect, run all again; a finished run is read from Drive and an unfinished one continues from its `last.ckpt`.

In [ ]:
import importlib
import itertools
import json
import sys

import pandas as pd

RUN20 = "--mode pretrain --tune --num_epochs 20"  # all the images, the full k-NN bank, val only (no test)
RUN100 = "--mode pretrain --tune --num_epochs 100"
CHOICE_FILE = f"{OUT}/tuning_choice.json"
choices = json.load(open(CHOICE_FILE))
assert "dae_h" in choices, "run colab_dae_h.ipynb to the end first: its choice is the base here"
base = {k: v for k, v in choices.get("dae_h_final", choices["dae"]).items() if k != "num_epochs"}
print("base:", base)


def flags_of(params):
    """The train.py flags of one DAE."""
    p = dict(params)
    return f"--channels {p.pop('conv1')} {p.pop('conv2')} " + " ".join(f"--{k} {v}" for k, v in p.items())


def folder_of(flags):
    """The run folder train.py uses for these flags (run_info.folder_name)."""
    import config, utils, run_info
    for module in (config, utils, run_info):
        importlib.reload(module)
    sys.argv = ["train.py", *flags.split(), "--output_dir", OUT]
    return f"{OUT}/{run_info.folder_name(utils.parse_args().mode)}"


def train(flags):
    """One run of train.py (skipped if it finished before): its val numbers and its folder."""
    if not os.path.exists(f"{folder_of(flags)}/results.json"):
        !python train.py {flags} --output_dir "{OUT}"
    return read(flags)


def read(flags):
    """The val numbers of a finished run, or None if it has not finished."""
    folder = folder_of(flags)
    if not os.path.exists(f"{folder}/results.json"):
        return None
    with open(f"{folder}/results.json") as f:
        return {**json.load(f), "run": os.path.basename(folder)}

## Which part
`PART = 2`: the 3 runs with seed 1.

In [ ]:
PART = 2  # seed 1: top 1, 2 and 3

## Stage A: last checks, 20 epochs
One change at a time from `base`; `base` itself was trained in `colab_dae_h.ipynb` and is read from Drive.

In [ ]:
checks = {"base": {}, "h 128": {"latent_dim": 128}, "h 64": {"latent_dim": 64},
          "+ gaussian 0.1": {"noise_type2": "gaussian", "noise_level2": 0.1},
          "+ gaussian 0.2": {"noise_type2": "gaussian", "noise_level2": 0.2},
          "batch 256": {"batch_size": 256}, "weight decay 0.05": {"wd": 0.05}}
if PART in (0, 1):
    rows = []
    for name, change in checks.items():
        r = train(f"{RUN20} {flags_of({**base, **change})}")
        rows.append({"check": name, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
    display(pd.DataFrame(rows).sort_values("val_knn", ascending=False).round(4))

## The top 3
Every full-data DAE run of 20 epochs so far, read from Drive (all of stage A must be done). The 3 best by val k-NN go on to stage B.

In [ ]:
b0 = {"noise_type": "salt_pepper", "noise_level": 0.5, "patch_size": 8, "conv1": 32, "conv2": 64,
      "latent_dim": 256, "dropout": 0.0, "augment": "flip", "batch_size": 512, "optimizer": "adamw",
      "activation": "gelu", "lr": 0.001, "wd": 0.01, "scheduler": "none", "noise_type2": "none", "noise_level2": 0.0}
h_winner = {k: v for k, v in choices["dae_h"].items() if k != "num_epochs"}  # stage 1 of colab_dae_h.ipynb
candidates = ([{**b0, "dropout": d, "lr": lr} for d, lr in itertools.product([0.0, 0.1, 0.2, 0.3], [0.001, 0.0005])]
              + [{**b0, "latent_dim": h, "dropout": d} for h, d in itertools.product([256, 384, 1024], [0.3, 0.4, 0.5])]
              + [{**h_winner, **c} for c in ({"noise_level": 0.6}, {"noise_level": 0.7}, {"loss": "l1"}, {"scheduler": "cosine"},
                                              {"noise_level": 0.3, "noise_level_end": 0.7})]
              + [{**base, **c} for c in checks.values()])
rows, missing = [], []
for params in candidates:
    r = read(f"{RUN20} {flags_of(params)}")
    if r is None:
        missing.append(params)
    else:
        rows.append({"params": params, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
assert not [p for p in missing if any(p == {**base, **c} for c in checks.values())], "stage A is not done yet"
all20 = pd.DataFrame(rows).drop_duplicates(subset="run").sort_values("val_knn", ascending=False)
display(all20[["val_knn", "val_loss", "run"]].head(10).round(4))
top3 = all20["params"].head(3).tolist()

## Stage B: the top 3, 100 epochs, seeds 42, 1 and 2
9 runs; `PART` 1, 2 or 3 takes every third one.

In [ ]:
tasks = [(i, seed) for i in range(3) for seed in (42, 1, 2)]
for i, seed in (tasks[PART - 1::3] if PART else tasks):
    params = {**top3[i], **({} if seed == 42 else {"seed": seed})}
    print(f"top {i + 1}, seed {seed}")
    train(f"{RUN100} {flags_of(params)}")

## The choice
All 9 runs, read from Drive (all three runtimes must be done): the mean and standard deviation over the 3 seeds. The best mean is saved as `"dae"`.

In [ ]:
rows = []
for i, seed in tasks:
    r = read(f"{RUN100} {flags_of({**top3[i], **({} if seed == 42 else {'seed': seed})})}")
    if r is None:  # not finished yet: left out of the table
        print(f"not finished yet: top {i + 1}, seed {seed}")
        continue
    rows.append({"top": i + 1, "seed": seed, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
runs100 = pd.DataFrame(rows)
display(runs100.round(4))
summary = runs100.groupby("top")["val_knn"].agg(["mean", "std", "min", "max"]).round(4)
display(summary)
if len(rows) == len(tasks):  # the choice only when all 9 runs are done
    best = int(summary["mean"].idxmax())
    choices.setdefault("dae_h_final", choices["dae"])  # the choice of colab_dae_h.ipynb, kept
    choices["dae"] = {**top3[best - 1], "num_epochs": 100}
    with open(CHOICE_FILE, "w") as f:
        json.dump(choices, f, indent=1)
    print(f"top {best}:", choices["dae"])